# 02  Dataset-level uncertainty indicators and robustness checks

Runs every analysis from the saved feature tables in `outputs/features/`, so the raw data are not
needed. All tables are written to `outputs/tables/`.

| Step | Output |
|---|---|
| Indicators $D_{\mathrm{hull}}$, $D_\Sigma$, $B$ and $\rho_\Sigma$ for every assembly state | `metrics.csv` |
| Recording-level bootstrap, jackknife of $D_{\mathrm{hull}}$ and exact permutation test | `inference.csv` |
| Sensitivity of $D_{\mathrm{hull}}$ to the outlier-screening threshold | `screening_sweep.csv`, `screening_summary.csv` |
| ISO 5725 style variance components | `variance_components.csv` |
| Task check: separation $\Delta$ and range $\delta$ | `task_check.csv` |
| Full-signal vs segmented dispersion | `sample_size_check.csv`, `subsample_draws.npz` |
| Synthetic check of $\rho_\Sigma$ and the permutation test | `synthetic_check.csv` |

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

import uncertainty_core as uc

OUTPUT_DIR = Path("outputs")
cfg = uc.Config(output_dir=OUTPUT_DIR)
store = uc.FeatureStore.load(cfg.subdir("features"))
tables = cfg.subdir("tables")
pd.set_option("display.float_format", lambda v: f"{v:.4g}")
uc.run_self_tests()
print(f"{len(store.full)} full-signal rows, {len(store.segmented)} segment rows")

All self-tests passed.
160 full-signal rows, 1600 segment rows


## 1. Indicators for every assembly state

An assembly state is one intervention case (C1 to C4) within one test session (T1 to T4).

* `D_hull` $=\sqrt{\mathrm{Area}(H)}$, with the 85 % outlier screening applied to the segmented sets only
* `D_sigma` $=(\det\hat\Sigma)^{1/4}$
* `B` $=\lVert\boldsymbol{\mu}_{C_j}-\boldsymbol{\mu}_{C_1}\rVert_2$, the bias (centroid shift) relative to C1 of the same test session, from all points
* `rho_sigma` $=B/D_{\Sigma,\mathrm{pool}}$, the effect size, with $\hat\Sigma_{\mathrm{pool}}=\tfrac12(\hat\Sigma_{C_1}+\hat\Sigma_{C_j})$

In [2]:
metrics = uc.MetricTable(cfg, store).build()
metrics.to_csv(tables / "metrics.csv", index=False)
for lv in uc.LEVELS:
    print(f"--- {lv} ---")
    display(metrics[(metrics.level == lv) & (metrics.case != "C1")]
            [["health", "test", "case", "D_hull", "D_sigma", "B", "rho_sigma"]])

--- full ---


,health,test,case,D_hull,D_sigma,B,rho_sigma
1,healthy,T1,C2,0.0001249,6.91e-05,0.0003029,2.591
2,healthy,T1,C3,0.000162,9.052e-05,0.000196,1.518
3,healthy,T1,C4,0.0002321,0.0001323,0.0002507,1.438
5,healthy,T2,C2,0.0003238,0.0001823,0.0001887,1.322
6,healthy,T2,C3,0.00024,0.0001325,0.0002796,2.079
7,healthy,T2,C4,0.0002701,0.000155,0.0004959,3.309
9,healthy,T3,C2,9.932e-05,5.798e-05,0.0002981,2.757
10,healthy,T3,C3,0.0001396,7.492e-05,0.0002505,2.127
11,healthy,T3,C4,0.000112,6.261e-05,0.0003264,2.856
13,healthy,T4,C2,0.0001412,7.321e-05,0.0001011,1.446


--- segmented ---


,health,test,case,D_hull,D_sigma,B,rho_sigma
33,healthy,T1,C2,0.0009584,0.0002774,0.0005066,1.505
34,healthy,T1,C3,0.001242,0.0003921,0.0002677,0.6817
35,healthy,T1,C4,0.001031,0.000311,0.0007452,2.081
37,healthy,T2,C2,0.001238,0.0004039,0.0004081,1.118
38,healthy,T2,C3,0.0009485,0.0003007,0.0003551,1.147
39,healthy,T2,C4,0.001398,0.0004154,0.0009056,2.446
41,healthy,T3,C2,0.00108,0.0003359,0.000488,1.234
42,healthy,T3,C3,0.001173,0.0003536,0.0002978,0.7402
43,healthy,T3,C4,0.0009185,0.0003431,0.0005626,1.409
45,healthy,T4,C2,0.0009624,0.0003061,5.441e-05,0.1749


## 2. Bootstrap, jackknife and permutation test

Recordings are the unit of repetition. The permutation test of "no intervention effect" relabels
the 10 recordings of C1 and Cj in all 252 ways; a bias is considered detected when $p < 0.05$.

In [3]:
inference = uc.Inference(cfg, store).run()
inference.to_csv(tables / "inference.csv", index=False)
cols = ["health", "test", "case", "level", "B_lo", "B_hi", "rho_sigma_lo", "rho_sigma_hi", "perm_p",
        "D_hull_jk_se", "rho_sigma_valid_frac"]
display(inference[inference.case != "C1"][cols])

,health,test,case,level,B_lo,B_hi,rho_sigma_lo,rho_sigma_hi,perm_p,D_hull_jk_se,rho_sigma_valid_frac
1,healthy,T1,C2,full,0.0001478,0.0004888,1.764,5.639,0.02381,5.181e-05,0.82
2,healthy,T1,C3,full,7.594e-05,0.0003745,0.8335,3.825,0.127,4.745e-05,0.822
3,healthy,T1,C4,full,0.0001826,0.0005417,1.207,4.464,0.3413,6.022e-05,0.8
5,healthy,T2,C2,full,6.246e-05,0.0003526,0.575,4.436,0.1429,0.0001635,0.8155
6,healthy,T2,C3,full,0.0001073,0.0005469,1.705,4.385,0.01587,0.0001106,0.811
7,healthy,T2,C4,full,0.0002648,0.000867,2.898,7.16,0.007937,0.0001128,0.8195
9,healthy,T3,C2,full,0.0001556,0.0005813,2.42,6.319,0.01587,4.465e-05,0.83
10,healthy,T3,C3,full,0.0001367,0.0005268,1.865,4.896,0.07937,5.313e-05,0.8055
11,healthy,T3,C4,full,0.0001407,0.0006263,2.087,6.748,0.01587,5.034e-05,0.817
13,healthy,T4,C2,full,3.985e-05,0.0001803,0.6375,3.803,0.1508,4.571e-05,0.814


## 3. Outlier-screening threshold sweep (all segmented assembly states)

In [4]:
sweep_runner = uc.ScreeningSweep(cfg, store)
sweep = sweep_runner.run()
sweep_summary = sweep_runner.summary(sweep)
sweep.to_csv(tables / "screening_sweep.csv", index=False)
sweep_summary.to_csv(tables / "screening_summary.csv", index=False)
display(sweep_summary)

,pct,D_hull_change_median_pct,D_hull_change_max_abs_pct,n_assembly_states
0,70,-6.289,20.93,32
1,80,0,10.81,32
2,85,0,0,32
3,90,0,10.43,32
4,95,2.284,11.43,32
5,100,10.09,30.56,32


## 4. ISO 5725 style variance components

Nested model: test session > assembly state (case within test) > recording > [segment].
$s_r$ is the repeatability standard deviation (recording to recording within one assembly state),
$s_R$ the reproducibility standard deviation (adding assembly-state and session components).

In [5]:
vc = uc.VarianceComponents(cfg, store).run()
vc.to_csv(tables / "variance_components.csv", index=False)
display(vc)

,level,health,feature,mean,var_test,var_assembly,var_recording,s_r,s_R,ratio_sR_sr,ICC_assembly,ICC_test,negative_set_to_zero,var_segment
0,full,healthy,A1,0.0007697,5.534e-09,8.677e-09,4.757e-08,0.0002181,0.0002486,1.14,0.1404,0.08957,,NaN
1,full,healthy,A2,0.0005954,3.839e-09,4.543e-09,6.401e-09,8.001e-05,0.0001216,1.52,0.3073,0.2597,,NaN
2,full,damaged,A1,0.01426,4.89e-06,3.083e-06,5.646e-06,0.002376,0.00369,1.553,0.2264,0.3591,,NaN
3,full,damaged,A2,0.00262,3.207e-06,8.778e-08,2.361e-07,0.0004859,0.001879,3.867,0.02486,0.9083,,NaN
4,segmented,healthy,A1,0.001688,7.449e-09,3.623e-08,2.717e-08,0.0004261,0.0004746,1.114,0.1609,0.03307,,1.544e-07
5,segmented,healthy,A2,0.001424,1.071e-08,3.204e-08,4.451e-09,0.000299,0.0003636,1.216,0.2424,0.08102,,8.498e-08
6,segmented,damaged,A1,0.01646,7.45e-06,2.92e-06,3.617e-06,0.002407,0.00402,1.67,0.1807,0.4609,,2.175e-06
7,segmented,damaged,A2,0.004017,1.614e-06,3.556e-07,3.509e-07,0.00101,0.001729,1.712,0.119,0.5398,,6.692e-07


## 5. Task check: separation between the health states and range within each health state

`Delta` is the smallest distance between a healthy and a damaged assembly-state centroid.
`delta_healthy` and `delta_damaged` are the largest distances between two centroids of the same
health state; `delta_healthy` is used for fault detection and `delta_damaged` for tasks within the
damaged state. The bootstrap interval of $\Delta/\delta$ gives the outcome: negligible if it lies
above one, ambiguous if it contains one, and influential if it lies below one.

In [6]:
tc = uc.TaskCheck(cfg, store).run()
tc.to_csv(tables / "task_check.csv", index=False)
display(tc)

,level,quantity,estimate,boot_lo,boot_hi
0,full,Delta,0.006409,0.005664,0.007065
1,full,between_max,0.01797,0.01704,0.01991
2,full,between_median,0.0139,0.01334,0.0146
3,full,delta_healthy,0.0006191,0.0004725,0.0009751
4,full,delta_damaged,0.01108,0.01008,0.0132
5,full,Delta_over_delta_healthy,10.35,6.059,13.94
6,full,Delta_over_delta_damaged,0.5787,0.4494,0.6782
7,segmented,Delta,0.007563,0.006099,0.009198
8,segmented,between_max,0.02029,0.01958,0.02131
9,segmented,between_median,0.01566,0.01452,0.01609


## 6. Full-signal vs segmented dispersion: sample size or real difference?

In [7]:
ss, draws = uc.SampleSizeCheck(cfg, store).run()
ss.to_csv(tables / "sample_size_check.csv", index=False)
np.savez_compressed(tables / "subsample_draws.npz", **{"_".join(k): v for k, v in draws.items()})
display(ss)
print("Median ratio segmented/full:  hull =", round(ss.ratio_hull.median(), 2),
      "  covariance =", round(ss.ratio_sigma.median(), 2))

,health,test,case,D_hull_full,D_hull_seg,D_sigma_full,D_sigma_seg,ratio_hull,ratio_sigma,sub5_median,sub5_lo,sub5_hi,full_D_percentile_in_sub5
0,healthy,T1,C1,0.0002721,0.001241,0.000141,0.0003863,4.562,2.74,0.000604,0.0002942,0.0009416,1.65
1,healthy,T1,C2,0.0001249,0.0009584,6.91e-05,0.0002774,7.672,4.015,0.000413,0.0001855,0.0006981,0.2
2,healthy,T1,C3,0.000162,0.001242,9.052e-05,0.0003921,7.664,4.331,0.0006001,0.0003088,0.0009566,0.1
3,healthy,T1,C4,0.0002321,0.001031,0.0001323,0.000311,4.441,2.352,0.0004665,0.0002336,0.0007598,2.4
4,healthy,T2,C1,9.619e-05,0.001173,5.22e-05,0.0003176,12.19,6.085,0.0004222,0.0002001,0.0008369,0
5,healthy,T2,C2,0.0003238,0.001238,0.0001823,0.0004039,3.825,2.216,0.0006069,0.0002858,0.001013,4.45
6,healthy,T2,C3,0.00024,0.0009485,0.0001325,0.0003007,3.951,2.27,0.0004609,0.0002307,0.000756,3.1
7,healthy,T2,C4,0.0002701,0.001398,0.000155,0.0004154,5.176,2.68,0.0006132,0.0002987,0.001082,1.45
8,healthy,T3,C1,0.0002576,0.001516,0.000137,0.0004367,5.886,3.187,0.0006483,0.0002951,0.001104,1.6
9,healthy,T3,C2,9.932e-05,0.00108,5.798e-05,0.0003359,10.87,5.792,0.0005054,0.0002529,0.0008275,0


Median ratio segmented/full:  hull = 4.3   covariance = 2.39


## 7. Synthetic check of the effect size and the permutation test

In [8]:
synth = uc.SyntheticCheck(cfg).run()
synth.to_csv(tables / "synthetic_check.csv", index=False)
display(synth)

,n_per_case,m_over_sigma,P_rho_sigma_gt_1,P_perm_p_lt_005
0,5,0,0.342,0.045
1,5,0.25,0.403,0.067
2,5,0.5,0.456,0.077
3,5,1,0.672,0.214
4,5,1.5,0.871,0.493
5,5,2,0.965,0.708
6,5,3,1,0.97
7,50,0,0,NaN
8,50,0.25,0,NaN
9,50,0.5,0.01,NaN


In [9]:
print("Tables written:")
for p in sorted(tables.iterdir()):
    print("  ", p.name)

Tables written:
   inference.csv
   metrics.csv
   sample_size_check.csv
   screening_summary.csv
   screening_sweep.csv
   subsample_draws.npz
   synthetic_check.csv
   task_check.csv
   variance_components.csv
